In [1]:
from multiprocess import Pool
import itertools
import json
import re
import numpy as np
from datasets import load_dataset

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from datasets import load_dataset

ds = load_dataset("mesolitica/pseudolabel-science-large-v3-timestamp")

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [29]:
df_ = ds['en'].to_pandas().to_dict(orient = 'records')

In [25]:
from glob import glob

files = sorted(glob('science-chunk/*.mp3'))
len(files)

916956

In [26]:
df_[0]['audio_filename'].replace('chunk/', 'science-chunk/')

'science-chunk/mp3-16k-0-0_001.mp3'

In [31]:
import os
from tqdm import tqdm
import re

df = []
for i in tqdm(range(len(df_))):
    new_f = df_[i]['audio_filename'].replace('chunk/', 'science-chunk/')
    if os.path.exists(new_f):
        df_[i]['audio_filename'] = new_f
        df.append((i, df_[i]))

len(df)

100%|██████████| 850638/850638 [00:06<00:00, 141733.83it/s]


850638

In [32]:
df[0]

(0,
 {'new_text': "<|startoftranscript|><|en|><|transcribe|><|0.00|> insightful reporting and engaging storytelling with deep roots in television and radio.<|5.96|><|6.48|> She's the former host of the national news program Here and Now, and she continues to host the<|11.82|><|11.82|> podcast Truth Be Told, which recently launched a new season called She Has a Name. Tanya,<|19.34|><|19.46|> welcome to the Minor Consult. It's great to have you here today. It's so great to be here,<|24.54|><|24.54|> Dean. I'm so excited. So, Tanya, you're in the news business, so let's start with your latest<|29.96|><|endoftext|>",
  'audio_filename': 'science-chunk/mp3-16k-0-0_001.mp3'})

In [33]:
def loop(rows):
    rows, _ = rows

    selected = []
    for r in tqdm(rows):
        try:
            with open(f'science-force-alignment/{r[0]}.json') as fopen:
                d = json.load(fopen)
        except:
            continue
        text = ' '.join([d_['text'] for d_ in d])

        if len(text) > 600:
            continue

        if d[0]['start'] > 5:
            continue

        failed = False
        for i in range(len(d)):
            if i > 0 and (d[i]['start'] - d[i - 1]['end']) > 3:
                failed = True
                break

            if (d[i]['end'] - d[i]['start']) > 1.5:
                failed = True
                break

        if failed:
            continue

        selected.append((r[1], d))
    return selected

In [34]:
filtered = loop((df[:10], 0))

100%|██████████| 10/10 [00:00<00:00, 8806.01it/s]


In [35]:
len(filtered)

8

In [36]:
filtered = multiprocessing(df, loop, cores = 30)

100%|██████████| 18/18 [00:00<00:00, 3923.37it/s]


In [37]:
len(filtered) / len(df), len(filtered), len(df)

(0.6057241740905062, 515252, 850638)

In [38]:
from collections import defaultdict
import os

audio_names = defaultdict(list)
for r in tqdm(filtered):
    i = os.path.split(r[0]['audio_filename'])[1].split('_')[0]
    i = '-'.join(i)
    audio_names[i].append(r)

100%|██████████| 515252/515252 [00:00<00:00, 548575.20it/s]


In [39]:
keys = list(audio_names.keys())
len(keys)

11346

In [40]:
os.path.split(r[0]['audio_filename'])[1].split('_')[1]

'205.mp3'

In [41]:
keys = list(audio_names.keys())

group = []
for k in tqdm(keys):
    s = sorted(audio_names[k], key = lambda x: int(os.path.split(x[0]['audio_filename'])[-1].split('_')[1].replace('.mp3', '')))
    temp = [s[0]]
    previous = int(s[0][0]['audio_filename'].split('-')[-1].replace('.mp3', ''))
    final = False
    for s_ in s[1:]:
        i = int(s_[0]['audio_filename'].split('-')[-1].replace('.mp3', ''))
        if previous + 1 == i:
            final = False
            temp.append(s_)
            previous += 1
        else:
            final = True
            group.append(temp)
            temp = [s_]
            previous = i
    
    if not final:
        group.append(temp)

100%|██████████| 11346/11346 [00:01<00:00, 10003.52it/s]


In [42]:
len(group)

177228

In [43]:
group = [(i, group[i]) for i in range(len(group))]

In [44]:
# !rm -rf malaysian-whole malaysian-segment
!mkdir science-whole
!mkdir science-segment

mkdir: cannot create directory ‘science-whole’: File exists
mkdir: cannot create directory ‘science-segment’: File exists


In [45]:
import copy
import soundfile as sf
import librosa
import numpy as np
import gc

def loop(group):
    group, _ = group
    combine_all = []
    for g in tqdm(group):
        i = g[0]
        g = g[1]
        audio_files = []
        timestamps = []
        last_timestamp = 0
        for g_ in g:
            audio_files.append(g_[0]['audio_filename'])
            timestamp = copy.deepcopy(g_[1])
            for k in range(len(timestamp)):
                timestamp[k]['start'] += last_timestamp
                timestamp[k]['end'] += last_timestamp
            timestamps.extend(timestamp)
            last_timestamp = timestamp[-1]['end']
    
        word_level = []
        for t in timestamps:
            start = t['start']
            w = t['text']
            end = t['end']
            word_level.append(f"<|{start:.2f}|> {w}<|{end:.2f}|>")
        
        segments, temp = [], [timestamps[0]]
        last_t = timestamps[0]['end']
        for c_ in timestamps[1:]:
            if ((c_['start'] - last_t) > 0.4):
                segments.append(temp)
                temp = []
    
            last_t = c_['end']
            temp.append(c_)
    
        if len(temp):
            segments.append(temp)
    
        segment_level = []
        for s in segments:
            start = s[0]['start']
            end = s[-1]['end']
            w = ' '.join([c_['text'] for c_ in s])
            t = f"<|{start:.2f}|> {w}<|{end:.2f}|>"
            segment_level.append(t)
    
        y = [librosa.load(f, sr = 16000)[0] for f in audio_files]
        y = np.concatenate(y)
    
        audio_filename = f'science-whole/{i}.mp3'
        sf.write(audio_filename, y, 16000)
    
        segment_audio_filenames = []
        streaming_word_level = []
        for k, s in enumerate(segments):
            segment_audio_filename = f'science-segment/{i}-{k}.mp3'
            start = s[0]['start']
            end = s[-1]['end']
            y_ = y[int(start * 16000): int(end * 16000)]
            sf.write(segment_audio_filename, y_, 16000)
            segment_audio_filenames.append(segment_audio_filename)

            del y_
    
            word_level_ = []
            for t in s:
                start = t['start']
                w = t['text']
                end = t['end']
                word_level_.append(f"<|{start:.2f}|> {w}<|{end:.2f}|>")
            streaming_word_level.append(''.join(word_level_))
            
        word_level = ''.join(word_level)
    
        combine_all.append({
            'mode': 'whole',
            'level': 'segment',
            'texts': [''.join(segment_level)],
            'audio_filenames': [audio_filename],
        })
        combine_all.append({
            'mode': 'whole',
            'level': 'word',
            'texts': [''.join(word_level)],
            'audio_filenames': [audio_filename],
        })
        combine_all.append({
            'mode': 'streaming',
            'level': 'segment',
            'texts': segment_level,
            'audio_filenames': segment_audio_filenames,
        })
        combine_all.append({
            'mode': 'streaming',
            'level': 'word',
            'texts': streaming_word_level,
            'audio_filenames': segment_audio_filenames,
        })

        del y, timestamps, segments, segment_level, word_level, streaming_word_level
        
    return combine_all

In [46]:
combine_all = loop((group[:2], 0))

100%|██████████| 2/2 [00:01<00:00,  1.11it/s]


In [47]:
combine_all[0]

{'mode': 'whole',
 'level': 'segment',
 'texts': ["<|0.02|> insightful reporting and engaging storytelling<|2.96|><|3.44|> with deep roots in television and radio.<|5.82|><|6.78|> She's the former host of the national news program Here and Now,<|9.98|><|10.52|> and she continues to host the podcast Truth Be Told,<|14.18|><|14.62|> which recently launched a new season called She Has a Name.<|18.28|><|19.06|> Tanya, welcome to the Minor Consult. It's great to have you here today.<|22.26|><|23.36|> It's so great to be here, Dean. I'm so excited.<|25.70|><|27.08|> So, Tanya, you're in the news business, so let's start with your latest<|29.94|>"],
 'audio_filenames': ['science-whole/0.mp3']}

In [ ]:
combine_all = multiprocessing(group, loop, cores = 50)

 95%|█████████▍| 3364/3544 [1:22:51<03:23,  1.13s/it]

In [ ]:
print('a')

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(combine_all)

In [ ]:
dataset.push_to_hub('malaysia-ai/Malaysian-STT', 'science_english')